In [ ]:
import os
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# TOKEN DE GITHUB
os.environ["GITHUB_TOKEN"] = "I n g r e s a r    t o k e n    a c a"

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com",
    temperature=0.2
)

print("Modelo GitHub Models configurado correctamente")

In [ ]:
import pdfplumber
import re

# Ruta del archivo PDF del reglamento
# Se debe asegurar que el archivo esta en la misma carpeta que este notebook
ruta_pdf = "C:\\Reglamento_Academico_Duoc\\RES-VRA-03-2024-NUEVO-REGLAMENTO-ACADÉMICO63-1.pdf"

print("Extrayendo texto del PDF...")

# Abrir el PDF y leer pagina por pagina
texto_completo = ""
with pdfplumber.open(ruta_pdf) as pdf:
    for pagina in pdf.pages:
        texto = pagina.extract_text()
        if texto:
            texto_completo += texto + "\n"

print(f"Texto extraido: {len(texto_completo)} caracteres")

# Fragmentar el texto por articulos
# Se busca el patron "Articulo N°1", "Articulo N°2", etc.
patron = r'(Art[ií]culo\s+N°\s*\d+)'

# Dividir el texto usando el patron como punto de separacion
partes = re.split(patron, texto_completo)

# Crear una lista para guardar cada articulo como un fragmento
fragmentos = []

# Recorrer las partes y armar cada articulo con su titulo y contenido
for i in range(1, len(partes), 2):
    if i + 1 < len(partes):
        titulo = partes[i].strip()
        contenido = partes[i+1].strip()
        # Limitar a 600 caracteres para no hacer los fragmentos muy largos
        if len(contenido) > 600:
            contenido = contenido[:600] + "..."
        fragmentos.append(f"{titulo} {contenido}")

print(f"Fragmentos creados: {len(fragmentos)}")
print("\nEjemplo del primer fragmento:")
print(fragmentos[0][:300])

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Cargar el modelo que convierte texto a numeros (embeddings)
# Este modelo transforma cada fragmento en un vector de 384 dimensiones
print("Cargando modelo de embeddings...")
modelo = SentenceTransformer('all-MiniLM-L6-v2')

# Generar embeddings para todos los fragmentos
print("Generando embeddings...")
embeddings = modelo.encode(fragmentos)
embeddings = np.array(embeddings).astype('float32')

# Crear un indice FAISS para buscar los fragmentos mas similares
# FAISS permite busqueda rapida por similitud de vectores
dimension = embeddings.shape[1]
indice = faiss.IndexFlatL2(dimension)
indice.add(embeddings)

print(f"Indice creado con {indice.ntotal} vectores")

In [ ]:
def buscar_fragmentos(pregunta, k=3):
    """
    Busca los k fragmentos mas similares a la pregunta del usuario
    
    Parametros:
    - pregunta: texto de la pregunta en lenguaje natural
    - k: numero de fragmentos a recuperar (por defecto 3)
    
    Retorna:
    - lista con los fragmentos mas relevantes
    """
    # Convertir la pregunta a embedding usando el mismo modelo
    embedding_preg = modelo.encode([pregunta])
    embedding_preg = np.array(embedding_preg).astype('float32')
    
    # Buscar en FAISS los k vectores mas cercanos
    distancias, indices = indice.search(embedding_preg, k)
    
    # Recuperar los fragmentos correspondientes a los indices encontrados
    resultados = []
    for idx in indices[0]:
        if idx < len(fragmentos):
            resultados.append(fragmentos[idx])
    
    return resultados

In [ ]:
def responder_pregunta(pregunta):
    """
    Genera una respuesta usando LangChain y OpenAI
    
    El flujo es:
    1. Buscar fragmentos relevantes del reglamento
    2. Construir el contexto con esos fragmentos
    3. Crear los mensajes (SystemMessage y HumanMessage)
    4. Enviar al modelo y retornar la respuesta
    
    Parametros:
    - pregunta: texto de la pregunta en lenguaje natural
    
    Retorna:
    - respuesta: texto generado por el modelo
    - fragmentos: lista de fragmentos usados como contexto
    """
    
    # Paso 1: Buscar fragmentos relevantes
    fragmentos_relevantes = buscar_fragmentos(pregunta, k=3)
    
    # Paso 2: Construir el contexto uniendo los fragmentos
    contexto = "\n---\n".join(fragmentos_relevantes)
    
    # Paso 3: Crear el mensaje del sistema con las reglas de comportamiento
    mensaje_sistema = SystemMessage(content="""
Eres un asistente academico especializado en el Reglamento Academico de Duoc UC.

Reglas:
1. Responde SOLO con la informacion que esta en el contexto que se te entregara
2. Si la pregunta no tiene respuesta en el contexto, responde: "No encontre esa informacion en el reglamento academico. Te recomiendo consultar con tu Direccion de Carrera."
3. Cuando sea posible, cita el articulo, ejemplo: "Segun el Articulo 37..."
4. Se claro, amable y directo
    """)
    
    # Paso 4: Crear el mensaje del humano con el contexto y la pregunta
    mensaje_humano = HumanMessage(content=f"""
CONTEXTO (informacion del reglamento):
{contexto}

PREGUNTA DEL ESTUDIANTE:
{pregunta}

RESPUESTA:
    """)
    
    # Paso 5: Llamar al modelo y obtener respuesta
    respuesta = llm.invoke([mensaje_sistema, mensaje_humano])
    
    return respuesta.content, fragmentos_relevantes

In [ ]:
# Pregunta de prueba para verificar que todo funciona
pregunta = "Cual es la nota minima para aprobar"

print("="*50)
print("PREGUNTA:", pregunta)
print("="*50)

# Obtener respuesta
respuesta, fragmentos = responder_pregunta(pregunta)

# Mostrar respuesta
print("\nRESPUESTA:")
print(respuesta)

# Mostrar los fragmentos que se usaron como contexto
print("\n" + "="*50)
print("FRAGMENTOS UTILIZADOS COMO CONTEXTO:")
print("="*50)
for i, frag in enumerate(fragmentos):
    print(f"\n--- Fragmento {i+1} ---")
    print(frag[:400])

In [ ]:
def chat_interactivo():
    """
    Funcion que permite hacer preguntas de forma interactiva
    El usuario puede escribir preguntas y recibe respuestas hasta que escribe "salir"
    """
    print("\n" + "="*50)
    print("ASISTENTE DEL REGLAMENTO ACADEMICO DUOC UC")
    print("Escribe 'salir' para terminar")
    print("="*50 + "\n")
    
    while True:
        # Pedir pregunta al usuario
        pregunta = input("\nTu pregunta: ")
        
        # Verificar si quiere salir
        if pregunta.lower() == "salir":
            print("\nHasta luego")
            break
        
        # Verificar que no este vacia
        if not pregunta.strip():
            print("Por favor escribe una pregunta")
            continue
        
        print("\nBuscando respuesta...\n")
        
        # Generar respuesta
        respuesta, fragmentos = responder_pregunta(pregunta)
        
        # Mostrar respuesta
        print("="*50)
        print("RESPUESTA:")
        print("="*50)
        print(respuesta)
        print("\n" + "="*50)

# Ejecutar el chat interactivo
chat_interactivo()